<a href="https://colab.research.google.com/github/bindumadamanchi3/ai-upskilling-journey/blob/main/Real_Public_Dataset_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np

# Load directly — no download needed
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/titanic_test.csv'

# We'll use the Heart Disease dataset instead — more relevant to AI/ML work
url = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'
df = pd.read_csv(url)

print(df.shape)
print(df.columns.tolist())
print(df.head())

(303, 14)
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  


In [27]:
# Print the column guide
guide = {
    'age':      'Age in years',
    'sex':      '1=Male, 0=Female',
    'cp':       'Chest pain type: 0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic',
    'trestbps': 'Resting blood pressure (mm Hg)',
    'chol':     'Serum cholesterol (mg/dl)',
    'fbs':      'Fasting blood sugar > 120: 1=True, 0=False',
    'restecg':  'Resting ECG: 0=normal, 1=ST-T abnormality, 2=left ventricular hypertrophy',
    'thalach':  'Max heart rate achieved',
    'exang':    'Exercise induced angina: 1=Yes, 0=No',
    'oldpeak':  'ST depression induced by exercise',
    'slope':    'Slope of peak exercise ST segment: 0=upsloping, 1=flat, 2=downsloping',
    'ca':       'Number of major vessels coloured by fluoroscopy (0-4)',
    'thal':     'Thalassemia: 1=normal, 2=fixed defect, 3=reversible defect',
    'target':   '1=Heart disease present, 0=No heart disease'
}

for col, desc in guide.items():
    print(f"  {col:<12} {desc}")

  age          Age in years
  sex          1=Male, 0=Female
  cp           Chest pain type: 0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic
  trestbps     Resting blood pressure (mm Hg)
  chol         Serum cholesterol (mg/dl)
  fbs          Fasting blood sugar > 120: 1=True, 0=False
  restecg      Resting ECG: 0=normal, 1=ST-T abnormality, 2=left ventricular hypertrophy
  thalach      Max heart rate achieved
  exang        Exercise induced angina: 1=Yes, 0=No
  oldpeak      ST depression induced by exercise
  slope        Slope of peak exercise ST segment: 0=upsloping, 1=flat, 2=downsloping
  ca           Number of major vessels coloured by fluoroscopy (0-4)
  thal         Thalassemia: 1=normal, 2=fixed defect, 3=reversible defect
  target       1=Heart disease present, 0=No heart disease


In [28]:
#  Full missing value audit (Easy)
# Get a complete picture of missing data: null count, percentage missing, and dtype per column. Then check if this dataset is cleaner than Titanic.
# Expected: summary table of nulls per column
# Spoiler: this dataset is much cleaner than Titanic

null_count = df.isnull().sum()
null_pct = (null_count/len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'null_count' : null_count,
    'pct_missing' : null_pct,
    'dtype' : df.dtypes
}).sort_values('null_count', ascending = False)
print(missing_summary)

complete = df.dropna().shape[0]
print(f"\nComplete rows: {complete} / {len(df)}")

print(f"Missing value columns: {(null_count > 0).sum()}") # false + false + false + all columns nulls + .. = 0
#This dataset has 0 missing values — already clean on nulls


          null_count  pct_missing    dtype
age                0          0.0    int64
sex                0          0.0    int64
cp                 0          0.0    int64
trestbps           0          0.0    int64
chol               0          0.0    int64
fbs                0          0.0    int64
restecg            0          0.0    int64
thalach            0          0.0    int64
exang              0          0.0    int64
oldpeak            0          0.0  float64
slope              0          0.0    int64
ca                 0          0.0    int64
thal               0          0.0    int64
target             0          0.0    int64

Complete rows: 303 / 303
Missing value columns: 0


In [29]:
# Check for hidden data quality issues (Medium)
# No nulls doesn't mean no problems. Check for: duplicate rows, outliers in blood pressure and cholesterol, and invalid values in categorical columns (sex, cp, thal, target should only have specific values).
# 2a. Any duplicate rows?
# 2b. Any suspiciously extreme values in trestbps or chol?
# 2c. Are all categorical columns within expected ranges?

duplicates_count = df.duplicated().sum()
print(f"Number of Duplicate rows: {duplicates_count}")

df= df.drop_duplicates()
print(f"Total rows  after dropping duplicates: {df.shape}")

#outlier check using z-score. Removing values z_score > 3 and z_score < -3 as outliers. we can also use IQR method to remove outliers
for col in ['trestbps', 'chol', 'thalach', 'age']:
  mean = df[col].mean()
  std = df[col].std()
  z_score = (df[col] - mean) / std
  outliers = (abs(z_score) > 3).sum()
  print(f"{col}: mean={mean:.1f}, std={std:.1f}, "
          f"range=[{df[col].min()}, {df[col].max()}], "
          f"outliers={outliers}")

#categorical value validation
expected = {
    'sex':    [0, 1],
    'cp':     [0, 1, 2, 3],
    'target': [0, 1],
    'thal':   [0, 1, 2, 3]
}

print("\nCategorical validation:")
for col, valid in expected.items():
  actual = sorted(df[col].unique().tolist())
  ok = all(v in valid for v in actual)
  print(f"  {col}: {actual} — {'OK' if ok else 'PROBLEM'}")

Number of Duplicate rows: 1
Total rows  after dropping duplicates: (302, 14)
trestbps: mean=131.6, std=17.6, range=[94, 200], outliers=2
chol: mean=246.5, std=51.8, range=[126, 564], outliers=4
thalach: mean=149.6, std=22.9, range=[71, 202], outliers=1
age: mean=54.4, std=9.0, range=[29, 77], outliers=0

Categorical validation:
  sex: [0, 1] — OK
  cp: [0, 1, 2, 3] — OK
  target: [0, 1] — OK
  thal: [0, 1, 2, 3] — OK


In [30]:
# Statistical profile of all features (Easy)
# Print a full descriptive statistics table. Identify which numeric features have the widest spread (highest CV) and which are most skewed.
# 3a. df.describe() with all columns
# 3b. Coefficient of variation per numeric column (std/mean*100)
# 3c. Which column is most variable relative to its scale?


print(df.describe().round(2))

numeric_cols = df.select_dtypes(include = [np.number]).columns
cv = (df[numeric_cols].std()/df[numeric_cols].mean() * 100).round(2)
cv_sorted = cv.sort_values(ascending=False)
print("\nCoefficient of variation per numeric column:")
print(cv_sorted)

print(f"\nMost variable feature: {cv_sorted.idxmax()} and CV = {cv_sorted.loc[cv_sorted.idxmax()]}")


          age     sex      cp  trestbps    chol     fbs  restecg  thalach  \
count  302.00  302.00  302.00    302.00  302.00  302.00   302.00   302.00   
mean    54.42    0.68    0.96    131.60  246.50    0.15     0.53   149.57   
std      9.05    0.47    1.03     17.56   51.75    0.36     0.53    22.90   
min     29.00    0.00    0.00     94.00  126.00    0.00     0.00    71.00   
25%     48.00    0.00    0.00    120.00  211.00    0.00     0.00   133.25   
50%     55.50    1.00    1.00    130.00  240.50    0.00     1.00   152.50   
75%     61.00    1.00    2.00    140.00  274.75    0.00     1.00   166.00   
max     77.00    1.00    3.00    200.00  564.00    1.00     2.00   202.00   

        exang  oldpeak   slope      ca    thal  target  
count  302.00   302.00  302.00  302.00  302.00  302.00  
mean     0.33     1.04    1.40    0.72    2.31    0.54  
std      0.47     1.16    0.62    1.01    0.61    0.50  
min      0.00     0.00    0.00    0.00    0.00    0.00  
25%      0.00     0.0

In [31]:
# Decode categorical columns (Easy)
# The dataset uses numeric codes for categorical variables. Create readable string versions of sex, cp, and target columns to make analysis more interpretable.
# sex:    1 → 'Male',   0 → 'Female'
# cp:     0 → 'Typical Angina', 1 → 'Atypical', 2 → 'Non-Anginal', 3 → 'Asymptomatic'
# target: 1 → 'Disease', 0 → 'No Disease'

df['sex_label'] = df['sex'].map({1: 'Male', 0: 'Female'})
df['cp_label'] = df['cp'].map({0:'Typical Angina', 1 : 'Atypical', 2 : 'Non-Anginal', 3 : 'Asymptomatic'})
df['target_label'] = df['target'].map({0:'No Disease', 1:'Disease'})

#verifying columns
print(df.columns.tolist())
print(df[['sex', 'sex_label', 'cp', 'cp_label', 'target', 'target_label']].head(8))
#models need numbers. Labels are for human analysis.


['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target', 'sex_label', 'cp_label', 'target_label']
   sex sex_label  cp        cp_label  target target_label
0    1      Male   3    Asymptomatic       1      Disease
1    1      Male   2     Non-Anginal       1      Disease
2    0    Female   1        Atypical       1      Disease
3    1      Male   1        Atypical       1      Disease
4    0    Female   0  Typical Angina       1      Disease
5    1      Male   0  Typical Angina       1      Disease
6    0    Female   1        Atypical       1      Disease
7    1      Male   1        Atypical       1      Disease


In [32]:
# Engineer new features (Medium)
# Create clinically meaningful features:

# age_group — Young (< 40), Middle (40–55), Senior (55–65), Elderly (65+)
# bp_category — Normal (< 120), Elevated (120–129), High (130+)
# high_chol — 1 if cholesterol > 240 (borderline high threshold), else 0
# heart_rate_reserve — max possible heart rate minus achieved: (220 - age) - thalach

# These features reflect real medical thresholds
# heart_rate_reserve: negative = exceeded expected max (unusual)

age_bins = [0, 40, 55, 65, 120]
age_labels=['Young', 'Middle', 'Senior', 'Elderly']
df['age_group'] = pd.cut(df['age'], bins = age_bins, labels = age_labels)

# bp_bins = [-np.inf, 119, 129, np.inf]
# bp_labels=['Normal', 'Elevated', 'High']
# df['bp_category'] = pd.cut(df['trestbps'], bins = bp_bins, labels = bp_labels)
# print(df.head())

#OR

def bp_cat(bp):
    if bp < 120:   return 'Normal'
    elif bp < 130: return 'Elevated'
    else:          return 'High'
df['bp_category'] = df['trestbps'].apply(bp_cat)

df['high_chol'] = (df['chol'] > 240).astype(int)

df['heart_rate_reserve'] = (220 - df['age']) - df['thalach']

print(df[['age', 'age_group', 'trestbps', 'bp_category',
          'chol', 'high_chol', 'heart_rate_reserve']].head(8))

print("\nAge group counts:")
print(df['age_group'].value_counts().sort_index())

print("\nBP category counts:")
print(df['bp_category'].value_counts())

print(f"\nPatients with high cholesterol: {df['high_chol'].sum()} "
      f"({df['high_chol'].mean():.1%})")

   age age_group  trestbps bp_category  chol  high_chol  heart_rate_reserve
0   63    Senior       145        High   233          0                   7
1   37     Young       130        High   250          1                  -4
2   41    Middle       130        High   204          0                   7
3   56    Senior       120    Elevated   236          0                 -14
4   57    Senior       120    Elevated   354          1                   0
5   57    Senior       140        High   192          0                  15
6   56    Senior       140        High   294          1                  11
7   44    Middle       120    Elevated   263          1                   3

Age group counts:
age_group
Young       18
Middle     133
Senior     118
Elderly     33
Name: count, dtype: int64

BP category counts:
bp_category
High        167
Elevated     75
Normal       60
Name: count, dtype: int64

Patients with high cholesterol: 151 (50.0%)


In [33]:
# Correlation analysis (Medium)
# Compute the correlation of every numeric feature with the target variable. Identify which features are most positively and negatively correlated with heart disease.
# Which features push toward disease? (positive correlation)
# Which features push away from disease? (negative correlation)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['target']]
print(numeric_cols)

#corrwith() computes correlation of every column against a single target Series — cleaner than a loop.
correlation = df[numeric_cols].corrwith(df['target']).round(2)
corr_sorted = correlation.sort_values(ascending=False)
print("\ncorrelation values sorted:")
print(corr_sorted)

#Top 3 features that are mostly or positively correlated
print("\nTop 3 positively correlated features:")
print(corr_sorted.head(3))

#Top 3 features that are mostly not correlated and negatively correlated
print("\nTop 3 negatively correlated features:")
print(corr_sorted.tail(3))

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'high_chol', 'heart_rate_reserve']

correlation values sorted:
cp                    0.43
thalach               0.42
slope                 0.34
restecg               0.13
fbs                  -0.03
chol                 -0.08
high_chol            -0.13
trestbps             -0.15
age                  -0.22
sex                  -0.28
thal                 -0.34
heart_rate_reserve   -0.36
ca                   -0.41
oldpeak              -0.43
exang                -0.44
dtype: float64

Top 3 positively correlated features:
cp         0.43
thalach    0.42
slope      0.34
dtype: float64

Top 3 negatively correlated features:
ca        -0.41
oldpeak   -0.43
exang     -0.44
dtype: float64


In [34]:
# Overall disease rate and by sex (Easy)
# Compute the overall heart disease rate. Then break it down by sex and age group.
# Overall disease rate
# Disease rate by sex
# Disease rate by age group

overall_rate = (df['target'].mean() * 100).round(2)
print(f"Overall heart disease rate: {overall_rate}%")

rate_by_sex = df.groupby('sex_label')['target'].mean().round(3) * 100
print("\nDisease rate by sex:")
print(rate_by_sex)

age_disease = df.groupby('age_group', observed=True).agg(
    count       = ('target', 'count'),
    disease_pct = ('target', 'mean')
).round(3)
age_disease['disease_pct'] = (age_disease['disease_pct'] * 100).round(1)
print(f"\nDisease rate by age group:")
print(age_disease)

Overall heart disease rate: 54.3%

Disease rate by sex:
sex_label
Female    75.0
Male      44.7
Name: target, dtype: float64

Disease rate by age group:
           count  disease_pct
age_group                    
Young         18         66.7
Middle       133         67.7
Senior       118         38.1
Elderly       33         51.5


In [38]:
# Chest pain type and disease (Medium)
# Chest pain type is the strongest single predictor. Analyse disease rate by each chest pain type and explain the counterintuitive finding.
# 8a. Disease rate by cp_label
# 8b. Which chest pain type has the HIGHEST disease rate?
# 8c. Why is 'Asymptomatic' the most dangerous?


rate_by_cp = df.groupby('cp_label')['target'].mean().round(2) * 100
print ("Disease rate by cp_label:")
print(rate_by_cp)

print("\nFindings")
print("Atypical has the highest disease rate which is 82%")


Disease rate by cp_label:
cp_label
Asymptomatic      70.0
Atypical          82.0
Non-Anginal       79.0
Typical Angina    27.0
Name: target, dtype: float64

Findings
Atypical has the highest disease rate which is 82%


In [43]:
# Age + Sex combined analysis (Medium)
# Build a pivot table showing disease rate for every Age Group × Sex combination.
# Rows: age_group
# Columns: sex_label
# Values: disease rate as percentage

rate_by_age_sex = df.groupby(['age_group', 'sex']).agg(
    count = ('target', 'count'),
    disease_rate = ('target', 'mean')
).round(2)

rate_by_age_sex['disease_rate'] = rate_by_age_sex['disease_rate'] * 100
rate_pivot = rate_by_age_sex.unstack()
print("Pivot table showing disease rate for every Age Group × Sex combination:")
print(rate_pivot)

print(f"\nThe group with the most disease rate is Young Females and the rate is {rate_pivot.loc['Young', ('disease_rate', 0)]}")
#Always check counts alongside rates — a 100% disease rate in a group of 2 patients means nothing statistically. This is why ML models need minimum sample sizes per group.

Pivot table showing disease rate for every Age Group × Sex combination:
          count     disease_rate      
sex           0   1            0     1
age_group                             
Young         5  13        100.0  54.0
Middle       39  94         90.0  59.0
Senior       39  79         51.0  32.0
Elderly      13  20         92.0  25.0

The group with the most disease rate is young girls and the rate is 100.0


/tmp/ipykernel_722/2340868411.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rate_by_age_sex = df.groupby(['age_group', 'sex']).agg(


In [51]:
# Blood pressure, cholesterol and disease (Medium)
# Analyse disease rates across blood pressure categories and high cholesterol flag. Then check if the combination of both risk factors compounds the risk.
# 10a. Disease rate by bp_category
# 10b. Disease rate by high_chol
# 10c. Disease rate for patients with BOTH high BP AND high cholesterol

print(df.columns.tolist())

disease_by_bp = df.groupby('bp_category').agg(
    count = ('target', 'count'),
    disease_rate = ('target', 'mean')
).round(2)
disease_by_bp['disease_rate'] = disease_by_bp['disease_rate'] * 100
print("Disease rate by bp_category:")
print(disease_by_bp)

disease_by_chol = df.groupby('high_chol').agg(
    count = ('target', 'count'),
    disease_rate = ('target', 'mean')
).round(2)
disease_by_chol['disease_rate'] = disease_by_chol['disease_rate'] * 100
disease_by_chol.index = ['Normal Chol', 'High Chol']
print("\nDisease rate by high_chol:")
print(disease_by_chol)

disease_by_bp_chol = df.groupby(['bp_category','high_chol']).agg(
    count = ('target', 'count'),
    disease_rate = ('target', 'mean')
).round(2)
disease_by_bp_chol['disease_rate'] = disease_by_bp_chol['disease_rate'] * 100
print("\nDisease rate by bp_category & high_chol:")
print(disease_by_bp_chol)


['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target', 'sex_label', 'cp_label', 'target_label', 'age_group', 'bp_category', 'high_chol', 'heart_rate_reserve']
Disease rate by bp_category:
             count  disease_rate
bp_category                     
Elevated        75          53.0
High           167          52.0
Normal          60          62.0

Disease rate by high_chol:
             count  disease_rate
Normal Chol    151          61.0
High Chol      151          48.0

Disease rate by bp_category & high_chol:
                       count  disease_rate
bp_category high_chol                     
Elevated    0             38          61.0
            1             37          46.0
High        0             76          61.0
            1             91          45.0
Normal      0             37          62.0
            1             23          61.0


In [52]:
# Group by Age Group, Blood Pressure, and Cholesterol to isolate the confounder
confounder_check = df.groupby(['age_group', 'bp_category'])['target'].mean().round(2) * 100
print(confounder_check)

age_group  bp_category
Young      Elevated       40.0
           High           88.0
           Normal         60.0
Middle     Elevated       64.0
           High           69.0
           Normal         69.0
Senior     Elevated       44.0
           High           36.0
           Normal         38.0
Elderly    Elevated       43.0
           High           47.0
           Normal         71.0
Name: target, dtype: float64


/tmp/ipykernel_722/1224518378.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  confounder_check = df.groupby(['age_group', 'bp_category'])['target'].mean().round(2) * 100


In [59]:
# Max heart rate and disease (Medium)
# Max heart rate (thalach) is one of the strongest predictors. Analyse it across disease groups and age groups.
# 11a. Mean thalach for disease vs no disease patients
# 11b. Mean thalach by age group — does heart rate decline with age?
# 11c. Correlation between thalach and age

thalach_rate = df.groupby('target_label')['thalach'].mean().round(2)
print("Mean thalach for disease vs no disease patients:")
print(thalach_rate)

thalach_rate_by_age = df.groupby(['age_group', 'target_label'])['thalach'].mean().round(2)
print("\nMean thalach by age group:")
print(thalach_rate_by_age)

corr = df['thalach'].corr(df['age'])
print(f"\nCorrelation between thalach and age: {corr:.3f}")

corr_target = df['thalach'].corr(df['target'])
print(f"\nCorrelation between thalach and disease: {corr_target:.3f}")

Mean thalach for disease vs no disease patients:
target_label
Disease       158.38
No Disease    139.10
Name: thalach, dtype: float64

Mean thalach by age group:
age_group  target_label
Young      Disease         178.75
           No Disease      150.50
Middle     Disease         162.00
           No Disease      139.91
Senior     Disease         152.78
           No Disease      139.30
Elderly    Disease         139.65
           No Disease      131.75
Name: thalach, dtype: float64

Correlation between thalach and age: -0.395

Correlation between thalach and disease: 0.420


/tmp/ipykernel_722/591339942.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  thalach_rate_by_age = df.groupby(['age_group', 'target_label'])['thalach'].mean().round(2)


In [64]:
# Exercise angina and ST depression (Medium)
# exang (exercise-induced angina) and oldpeak (ST depression) are both measured during stress tests. Analyse their relationship with disease together.
# 12a. Disease rate for exang=1 vs exang=0
# 12b. Mean oldpeak for disease vs no disease
# 12c. Among patients WITH exercise angina, does oldpeak magnitude matter?

exang_disease = df.groupby('exang').agg(
    count = ('target', 'count'),
    disease_rate = ('target', 'mean')
).round(2)
exang_disease['disease_rate'] = exang_disease['disease_rate'] * 100
exang_disease.index = ['No Angina', 'Exercise Angina']
print("Disease rate for exang=1 vs exang=0: ")
print(exang_disease)

oldpeak_disease = df.groupby('target_label').agg(
    count = ('oldpeak', 'count'),
    disease_rate = ('oldpeak', 'mean')
).round(2)
print("\nMean oldpeak for disease vs no disease:")
print(oldpeak_disease)

with_angina = df[df['exang'] == 1]

oldpeak_summary = with_angina.groupby('target', observed=False)['oldpeak'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std'
).round(2)

print("\nOldpeak magnitude metrics for patients WITH exercise angina:")
print(oldpeak_summary)

Disease rate for exang=1 vs exang=0: 
                 count  disease_rate
No Angina          203          69.0
Exercise Angina     99          23.0

Mean oldpeak for disease vs no disease:
              count  disease_rate
target_label                     
Disease         164          0.59
No Disease      138          1.59

Oldpeak magnitude metrics for patients WITH exercise angina:
        count  mean  median   std
target                           
0          76  1.78     1.8  1.24
1          23  0.65     0.2  0.80
